RetailPulse 360

Notebook 03 — Product Taxonomy, Color & Cross-Sell (H&M)

Goal: Rossmann and our own store network gave us the "when" and "where" of demand. Neither
tells us anything about the "what" — which colors sell, how footwear categories should be
structured, or which products get bought together. This notebook uses real H&M fashion
retail data (articles.csv + transactions_train.csv) to ground those decisions in observed
behavior instead of guesswork, then adapts the patterns to footwear categories.

Input: H&M articles.csv, transactions_train.csv (attached directly via Kaggle Add Input)
Output: color_popularity.csv, cross_sell_pairs.csv, footwear_taxonomy_mapping.csv

In [1]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# 2. LOAD ARTICLES (PRODUCT METADATA)
# ============================================================

ARTICLES_PATH = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/articles.csv"

articles = pd.read_csv(ARTICLES_PATH)

print("Articles loaded successfully.")
print("Shape:", articles.shape)
articles.head()

Articles loaded successfully.
Shape: (105542, 25)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [3]:
# 3. DATA QUALITY — VALIDATE ARTICLES
# ============================================================

print("Total articles:", len(articles))

print("\nDuplicate article_id:", articles["article_id"].duplicated().sum())

print("\nMissing values (columns with any):")
missing = articles.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nUnique index_group_name values (gender/age division):")
print(articles["index_group_name"].value_counts())

print("\nUnique product_group_name values (top-level category):")
print(articles["product_group_name"].value_counts())

print("\nDoes 'Shoes' exist as a product_group_name?")
print(articles[articles["product_group_name"].str.contains("Shoe", case=False, na=False)]["product_group_name"].unique())

Total articles: 105542

Duplicate article_id: 0

Missing values (columns with any):
detail_desc    416
dtype: int64

Unique index_group_name values (gender/age division):
index_group_name
Ladieswear       39737
Baby/Children    34711
Divided          15149
Menswear         12553
Sport             3392
Name: count, dtype: int64

Unique product_group_name values (top-level category):
product_group_name
Garment Upper body       42741
Garment Lower body       19812
Garment Full body        13292
Accessories              11158
Underwear                 5490
Shoes                     5283
Swimwear                  3127
Socks & Tights            2442
Nightwear                 1899
Unknown                    121
Underwear/nightwear         54
Cosmetic                    49
Bags                        25
Items                       17
Furniture                   13
Garment and Shoe care        9
Stationery                   5
Interior textile             3
Fun                          2
Name: c

In [4]:
# 4. FILTER TO FOOTWEAR ARTICLES
# ============================================================
# H&M genuinely has a "Shoes" category — we use real footwear data
# directly rather than proxying from clothing categories.

shoes = articles[articles["product_group_name"] == "Shoes"].copy()

print("Footwear articles:", len(shoes))

print("\nindex_group_name breakdown (within Shoes):")
print(shoes["index_group_name"].value_counts())

print("\nproduct_type_name breakdown (within Shoes):")
print(shoes["product_type_name"].value_counts())

print("\ncolour_group_name breakdown (within Shoes) — top 15:")
print(shoes["colour_group_name"].value_counts().head(15))

Footwear articles: 5283

index_group_name breakdown (within Shoes):
index_group_name
Baby/Children    2239
Ladieswear       2093
Menswear          672
Divided           277
Sport               2
Name: count, dtype: int64

product_type_name breakdown (within Shoes):
product_type_name
Sneakers          1621
Boots             1028
Sandals            757
Other shoe         395
Ballerinas         372
Slippers           249
Heeled sandals     202
Pumps              188
Flat shoe          165
Flip flop          125
Wedge              113
Bootie              31
Heels               22
Flat shoes          10
Moccasins            4
Pre-walkers          1
Name: count, dtype: int64

colour_group_name breakdown (within Shoes) — top 15:
colour_group_name
Black              1684
White               534
Dark Blue           500
Yellowish Brown     318
Light Pink          305
Beige               212
Light Beige         170
Light Orange        114
Dark Beige          109
Red                  97
Silver    

In [5]:
# 5. MAP H&M CATEGORIES TO STYLO'S SCHEME
# ============================================================
# Gender/age mapping — 3 map directly; "Divided" is H&M's young-adult
# line (not officially gendered, but skews heavily toward young
# women's fashion in practice) — documented assumption, not a fact
# the data states directly.
gender_map = {
    "Ladieswear": "Women's",
    "Menswear": "Men's",
    "Baby/Children": "Kids",
    "Divided": "Women's",   # assumption — see note above
}
shoes["stylo_gender"] = shoes["index_group_name"].map(gender_map)

# Style mapping — Sports is deliberately NOT populated from H&M data.
# H&M's real "Sport" index has only 2 shoe rows (statistically nothing),
# and Sneakers (H&M's largest shoe category) are predominantly fashion/
# casual, not athletic — mapping them to Sports would misrepresent both
# categories. Sports color/style patterns will use documented industry
# assumptions instead, applied later, not derived from this dataset.
style_map = {
    "Sneakers": "Casual", "Boots": "Casual", "Sandals": "Casual",
    "Flip flop": "Casual", "Slippers": "Casual", "Flat shoe": "Casual",
    "Flat shoes": "Casual", "Moccasins": "Casual", "Other shoe": "Casual",
    "Pre-walkers": "Casual",
    "Ballerinas": "Formal", "Heeled sandals": "Formal", "Pumps": "Formal",
    "Wedge": "Formal", "Bootie": "Formal", "Heels": "Formal",
}
shoes["stylo_style"] = shoes["product_type_name"].map(style_map)

print("Gender mapping result:")
print(shoes["stylo_gender"].value_counts(dropna=False))
print("\nStyle mapping result:")
print(shoes["stylo_style"].value_counts(dropna=False))

Gender mapping result:
stylo_gender
Women's    2370
Kids       2239
Men's       672
NaN           2
Name: count, dtype: int64

Style mapping result:
stylo_style
Casual    4355
Formal     928
Name: count, dtype: int64


In [6]:
# 6. FINALIZE MAPPED DATASET & COMPUTE COLOR POPULARITY
# ============================================================
# Drop the 2 rows with no gender mapping — these are the "Sport"
# index_group rows we already decided not to use as real signal.

shoes_mapped = shoes.dropna(subset=["stylo_gender", "stylo_style"]).copy()
print("Rows after dropping unmapped:", len(shoes_mapped))
assert len(shoes_mapped) == 5281, "Expected 5281 rows (5283 - 2 unmapped)"

# Real color popularity, broken down by our Stylo category scheme
color_popularity = (
    shoes_mapped
    .groupby(["stylo_gender", "stylo_style", "colour_group_name"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["stylo_gender", "stylo_style", "article_count"], ascending=[True, True, False])
)

print("\nTop 5 colors per Gender x Style combination:")
for (gender, style), group in color_popularity.groupby(["stylo_gender", "stylo_style"]):
    print(f"\n{gender} / {style}:")
    print(group.head(5)[["colour_group_name", "article_count"]].to_string(index=False))

Rows after dropping unmapped: 5281

Top 5 colors per Gender x Style combination:

Kids / Casual:
colour_group_name  article_count
            Black            388
        Dark Blue            370
            White            236
       Light Pink            193
  Yellowish Brown             89

Kids / Formal:
colour_group_name  article_count
            Black             36
       Light Pink             32
            White             31
        Dark Blue             23
    Bronze/Copper             19

Men's / Casual:
colour_group_name  article_count
            Black            240
  Yellowish Brown            102
        Dark Blue             89
            White             72
            Beige             30

Men's / Formal:
colour_group_name  article_count
            Black              1

Women's / Casual:
colour_group_name  article_count
            Black            746
            White            156
  Yellowish Brown             96
      Light Beige             78
         

In [7]:
# 6b. INVESTIGATE — WHY IS MEN'S FORMAL SO SPARSE?
# ============================================================
menswear_shoes = shoes[shoes["index_group_name"] == "Menswear"]

print("All Menswear shoe product_type_name counts:")
print(menswear_shoes["product_type_name"].value_counts())

All Menswear shoe product_type_name counts:
product_type_name
Sneakers      276
Other shoe    175
Boots         107
Sandals        55
Flip flop      25
Slippers       23
Flat shoes      9
Bootie          1
Flat shoe       1
Name: count, dtype: int64


In [8]:
# 6c. FINALIZE COLOR POPULARITY — DOCUMENT KNOWN GAPS
# ============================================================
# Two category combinations have no meaningful real signal in H&M data:
#   - Sports (any gender): H&M's "Sport" index has only 2 shoe rows total
#   - Men's / Formal: H&M's menswear catalog has no real dress-shoe
#     category (confirmed above — only 1 stray "Bootie" row)
# Both will use documented industry-standard color assumptions
# downstream, NOT figures from this dataset. Excluding them here keeps
# this output honestly labeled as real, data-derived signal only.

MIN_ARTICLES_FOR_SIGNAL = 10  # below this, we don't trust it as real signal

color_popularity_final = (
    color_popularity
    .groupby(["stylo_gender", "stylo_style"])
    .filter(lambda g: g["article_count"].sum() >= MIN_ARTICLES_FOR_SIGNAL)
)

covered_combos = color_popularity_final[["stylo_gender","stylo_style"]].drop_duplicates()
print("Gender x Style combinations with real H&M signal:")
print(covered_combos.to_string(index=False))

print("\nSaving as color_popularity.csv")
color_popularity_final.to_csv("color_popularity.csv", index=False)
print("Saved:", color_popularity_final.shape)

Gender x Style combinations with real H&M signal:
stylo_gender stylo_style
        Kids      Casual
        Kids      Formal
       Men's      Casual
     Women's      Casual
     Women's      Formal

Saving as color_popularity.csv
Saved: (178, 4)


In [9]:
# 7. LOAD TRANSACTIONS (MEMORY-EFFICIENT)
# ============================================================
# transactions_train.csv is ~31M rows. Default pandas dtypes waste
# memory here — article_id doesn't need 64-bit ints, and customer_id
# as raw 64-char strings is extremely wasteful repeated 31M times.
# We downcast dtypes and factorize customer_id into compact integer
# codes to make this manageable.

TRANSACTIONS_PATH = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/transactions_train.csv"

transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={
        "article_id": "int32",
        "price": "float32",
        "sales_channel_id": "int8",
    },
    parse_dates=["t_dat"],
)

# factorize customer_id: replaces the long hex string with a compact integer code
transactions["customer_code"], _ = pd.factorize(transactions["customer_id"])
transactions["customer_code"] = transactions["customer_code"].astype("int32")
transactions = transactions.drop(columns=["customer_id"])

print("Transactions loaded successfully.")
print("Shape:", transactions.shape)
print("Memory usage:", round(transactions.memory_usage(deep=True).sum() / 1e9, 2), "GB")
transactions.head()

Transactions loaded successfully.
Shape: (31788324, 5)
Memory usage: 0.67 GB


,t_dat,article_id,price,sales_channel_id,customer_code
0,2018-09-20,663713001,0.050831,2,0
1,2018-09-20,541518023,0.030492,2,0
2,2018-09-20,505221004,0.015237,2,1
3,2018-09-20,685687003,0.016932,2,1
4,2018-09-20,685687004,0.016932,2,1


In [10]:
# 8. DATA QUALITY — VALIDATE TRANSACTIONS
# ============================================================

print("Total transactions:", len(transactions))
print("Date range:", transactions["t_dat"].min(), "to", transactions["t_dat"].max())
print("Unique customers:", transactions["customer_code"].nunique())
print("Unique articles purchased:", transactions["article_id"].nunique())

print("\nMissing values:")
missing = transactions.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nExact duplicate rows (same customer, date, article, price, channel):")
print(transactions.duplicated().sum())

print("\nPrice sanity check:")
print(transactions["price"].describe())

print("\nsales_channel_id values:")
print(transactions["sales_channel_id"].value_counts())

print("\nReferential integrity — do all purchased article_ids exist in articles.csv?")
orphan_articles = ~transactions["article_id"].isin(articles["article_id"])
print("Orphan transaction rows (article not in catalog):", orphan_articles.sum())

Total transactions: 31788324
Date range: 2018-09-20 00:00:00 to 2020-09-22 00:00:00
Unique customers: 1362281
Unique articles purchased: 104547

Missing values:
None

Exact duplicate rows (same customer, date, article, price, channel):
2974905

Price sanity check:
count    3.178832e+07
mean     2.782928e-02
std      1.833207e-02
min      1.694915e-05
25%      1.581356e-02
50%      2.540678e-02
75%      3.388136e-02
max      5.915254e-01
Name: price, dtype: float64

sales_channel_id values:
sales_channel_id
2    22379862
1     9408462
Name: count, dtype: int64

Referential integrity — do all purchased article_ids exist in articles.csv?
Orphan transaction rows (article not in catalog): 0


In [11]:
# 8b. INVESTIGATE — WHY ARE THERE ~3M "DUPLICATE" ROWS?
# ============================================================
# H&M's transactions have no quantity column — each row represents
# one unit. If a customer bought 2 of the exact same item on the same
# day, that legitimately produces two identical rows, not a data error.

dupe_rows = transactions[transactions.duplicated(keep=False)]
print("Rows involved in duplication:", len(dupe_rows))
print("Unique customers involved:", dupe_rows["customer_code"].nunique())
print("Unique articles involved:", dupe_rows["article_id"].nunique())

# how many times does the MOST duplicated single (customer, article, date) combo repeat?
dupe_counts = transactions.groupby(
    ["customer_code", "article_id", "t_dat"]
).size()
print("\nMax repeat count for a single customer+article+date:", dupe_counts.max())
print("Distribution of repeat counts (how many times a combo repeats):")
print(dupe_counts.value_counts().head(10))

Rows involved in duplication: 5518813
Unique customers involved: 549707
Unique articles involved: 77985

Max repeat count for a single customer+article+date: 570
Distribution of repeat counts (how many times a combo repeats):
1     25857551
2      2418787
3       205350
4        63677
5        10207
6        10013
8         2429
7         1984
10        1300
9          967
Name: count, dtype: int64


In [12]:
# 8c. INVESTIGATE — HOW MANY CUSTOMERS SHOW EXTREME REPEAT BUYING?
# ============================================================
# The bulk of repeats (2-10x) look like genuine multi-unit purchases.
# But a max of 570 for a single customer+article+day is not normal
# retail behavior — likely a reseller/wholesale account. We check how
# widespread this is before deciding a cutoff.

extreme_threshold = 20  # more than 20 of the same item, same day = suspicious

extreme_combos = dupe_counts[dupe_counts > extreme_threshold]
print("Customer+article+date combos exceeding", extreme_threshold, "units:", len(extreme_combos))
print("Rows these account for:", extreme_combos.sum())
print("Unique customers involved:", 
      extreme_combos.reset_index()["customer_code"].nunique())
print("\nAs % of total transaction rows:", 
      round(extreme_combos.sum() / len(transactions) * 100, 3), "%")

Customer+article+date combos exceeding 20 units: 636
Rows these account for: 21024
Unique customers involved: 505

As % of total transaction rows: 0.066 %


In [13]:
# 8d. REMOVE EXTREME REPEAT-PURCHASE OUTLIERS
# ============================================================
# Only removing the specific (customer, article, date) combos that
# exceed the threshold — NOT the customers entirely, since their
# other, normal purchases are still legitimate data.

extreme_keys = extreme_combos.reset_index()[["customer_code", "article_id", "t_dat"]]
extreme_keys["is_extreme"] = True

transactions = transactions.merge(
    extreme_keys, on=["customer_code", "article_id", "t_dat"], how="left"
)
transactions = transactions[transactions["is_extreme"] != True].drop(columns=["is_extreme"])

print("Transactions after removing extreme outliers:", len(transactions))
print("Rows removed:", 31788324 - len(transactions))

Transactions after removing extreme outliers: 31767300
Rows removed: 21024


In [14]:
# 9. FILTER TO FOOTWEAR TRANSACTIONS & MEASURE VOLUME
# ============================================================
# Before deciding cross-sell scope, measure how much real shoe-purchase
# data actually exists — an informed decision beats a guess.

shoe_article_ids = set(shoes_mapped["article_id"])
shoe_transactions = transactions[transactions["article_id"].isin(shoe_article_ids)].copy()

print("Total shoe-purchase transactions:", len(shoe_transactions))
print("As % of all transactions:", round(len(shoe_transactions) / len(transactions) * 100, 2), "%")
print("Unique customers who bought shoes:", shoe_transactions["customer_code"].nunique())

# for cross-sell pairs to exist at all, we need customers who bought
# 2+ DIFFERENT shoe articles on the same day (a "basket")
basket_sizes = shoe_transactions.groupby(["customer_code", "t_dat"])["article_id"].nunique()
multi_shoe_baskets = basket_sizes[basket_sizes >= 2]

print("\nShoe-purchase 'baskets' (customer+date):", len(basket_sizes))
print("Baskets with 2+ different shoe articles (usable for cross-sell):", len(multi_shoe_baskets))

Total shoe-purchase transactions: 745266
As % of all transactions: 2.35 %
Unique customers who bought shoes: 269183

Shoe-purchase 'baskets' (customer+date): 523104
Baskets with 2+ different shoe articles (usable for cross-sell): 109597


In [15]:
# 10. BUILD CROSS-SELL PAIRS (MARKET BASKET ANALYSIS)
# ============================================================
# For every basket with 2+ different shoe articles, generate every
# pair of items bought together, then count how often each pair occurs
# across all baskets. Frequent pairs = real cross-sell signal.

from itertools import combinations

# keep only baskets that actually have 2+ different articles
multi_shoe_basket_keys = multi_shoe_baskets.index  # (customer_code, t_dat) pairs

basket_items = (
    shoe_transactions
    .set_index(["customer_code", "t_dat"])
    .loc[multi_shoe_basket_keys]
    .groupby(level=[0, 1])["article_id"]
    .apply(lambda x: sorted(set(x)))
)

print("Baskets to process:", len(basket_items))

pair_counts = {}
for items in basket_items:
    for pair in combinations(items, 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

print("Unique cross-sell pairs found:", len(pair_counts))

cross_sell_df = pd.DataFrame(
    [(a, b, count) for (a, b), count in pair_counts.items()],
    columns=["article_id_a", "article_id_b", "co_purchase_count"]
).sort_values("co_purchase_count", ascending=False)

print("\nTop 10 most frequent shoe-pair co-purchases:")
cross_sell_df.head(10)

Baskets to process: 109597
Unique cross-sell pairs found: 133533

Top 10 most frequent shoe-pair co-purchases:


,article_id_a,article_id_b,co_purchase_count
2885,734460001,734460002,337
3389,349301001,349301041,246
1588,606711003,606711005,171
30,682511001,709138001,166
3296,734460001,734460006,152
230,650672001,650672002,142
130,723874001,723874002,141
359,349301001,349301025,137
4767,727880001,727880003,131
736,808628001,808628002,129


In [16]:
# 11. INVESTIGATE — SAME PRODUCT (DIFFERENT COLOR) VS TRUE CROSS-SELL
# ============================================================
# H&M's article_id embeds product_code as a prefix. If two paired
# articles share the same product_code, the customer bought the SAME
# style in two colors/variants — not a cross-sell between different
# products. We check how much of our pair data this affects.

article_to_product = articles.set_index("article_id")["product_code"]

cross_sell_df["product_code_a"] = cross_sell_df["article_id_a"].map(article_to_product)
cross_sell_df["product_code_b"] = cross_sell_df["article_id_b"].map(article_to_product)
cross_sell_df["same_style_different_variant"] = (
    cross_sell_df["product_code_a"] == cross_sell_df["product_code_b"]
)

print("Total pairs:", len(cross_sell_df))
print("Same-style variant pairs:", cross_sell_df["same_style_different_variant"].sum())
print("True cross-style pairs:", (~cross_sell_df["same_style_different_variant"]).sum())

print("\nOf the top 10 by frequency, how many are same-style variants?")
print(cross_sell_df.head(10)["same_style_different_variant"].sum(), "out of 10")

print("\nTop 10 TRUE cross-style pairs (different products actually bought together):")
true_cross_sell = cross_sell_df[~cross_sell_df["same_style_different_variant"]]
true_cross_sell.sort_values("co_purchase_count", ascending=False).head(10)

Total pairs: 133533
Same-style variant pairs: 2464
True cross-style pairs: 131069

Of the top 10 by frequency, how many are same-style variants?
9 out of 10

Top 10 TRUE cross-style pairs (different products actually bought together):


,article_id_a,article_id_b,co_purchase_count,product_code_a,product_code_b,same_style_different_variant
30,682511001,709138001,166,682511,709138,False
8852,710056003,734460001,107,710056,734460,False
78,502224001,709138001,103,502224,709138,False
4822,745184004,755604001,103,745184,755604,False
1757,622958016,736489003,102,622958,736489,False
1789,631878001,734460001,80,631878,734460,False
856,574752003,655287002,77,574752,655287,False
6984,734460001,755604001,76,734460,755604,False
9705,581363001,586955001,76,581363,586955,False
4108,721468001,734460001,75,721468,734460,False


In [17]:
# 12. FINALIZE TRUE CROSS-SELL PAIRS
# ============================================================
# Excludes same-style color variants (Section 11). Also applies a
# minimum co-purchase count — a pair appearing only 1-2 times across
# 109K baskets is statistical noise, not a real pattern.

print("Co-purchase count distribution (true cross-style pairs only):")
print(true_cross_sell["co_purchase_count"].describe())
print("\nPercentiles:")
print(true_cross_sell["co_purchase_count"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

MIN_CO_PURCHASE = 5  # pairs below this are treated as noise, not real pattern

cross_sell_final = true_cross_sell[
    true_cross_sell["co_purchase_count"] >= MIN_CO_PURCHASE
].copy()

# enrich with readable info for downstream use
enrich_cols = ["product_type_name", "colour_group_name", "stylo_gender", "stylo_style"]
enrich_a = shoes_mapped.set_index("article_id")[enrich_cols].add_suffix("_a")
enrich_b = shoes_mapped.set_index("article_id")[enrich_cols].add_suffix("_b")

cross_sell_final = cross_sell_final.merge(enrich_a, left_on="article_id_a", right_index=True, how="left")
cross_sell_final = cross_sell_final.merge(enrich_b, left_on="article_id_b", right_index=True, how="left")

print("\nFinal cross-sell pairs (>=", MIN_CO_PURCHASE, "co-purchases):", len(cross_sell_final))
cross_sell_final.to_csv("cross_sell_pairs.csv", index=False)
print("Saved cross_sell_pairs.csv")

cross_sell_final.sort_values("co_purchase_count", ascending=False).head(10)

Co-purchase count distribution (true cross-style pairs only):
count    131069.000000
mean          1.715860
std           2.431733
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         166.000000
Name: co_purchase_count, dtype: float64

Percentiles:
0.50     1.0
0.75     2.0
0.90     3.0
0.95     4.0
0.99    11.0
Name: co_purchase_count, dtype: float64

Final cross-sell pairs (>= 5 co-purchases): 6422
Saved cross_sell_pairs.csv


,article_id_a,article_id_b,co_purchase_count,product_code_a,product_code_b,same_style_different_variant,product_type_name_a,colour_group_name_a,stylo_gender_a,stylo_style_a,product_type_name_b,colour_group_name_b,stylo_gender_b,stylo_style_b
30,682511001,709138001,166,682511,709138,False,Boots,Black,Women's,Casual,Boots,Black,Women's,Casual
8852,710056003,734460001,107,710056,734460,False,Sandals,Black,Women's,Casual,Sandals,Black,Women's,Casual
78,502224001,709138001,103,502224,709138,False,Boots,Black,Women's,Casual,Boots,Black,Women's,Casual
4822,745184004,755604001,103,745184,755604,False,Sandals,Black,Women's,Casual,Sandals,Black,Women's,Casual
1757,622958016,736489003,102,622958,736489,False,Sneakers,White,Women's,Casual,Sneakers,White,Women's,Casual
1789,631878001,734460001,80,631878,734460,False,Sandals,Black,Women's,Casual,Sandals,Black,Women's,Casual
856,574752003,655287002,77,574752,655287,False,Sneakers,White,Women's,Casual,Sneakers,White,Women's,Casual
9705,581363001,586955001,76,581363,586955,False,Ballerinas,Black,Women's,Formal,Ballerinas,Black,Women's,Formal
6984,734460001,755604001,76,734460,755604,False,Sandals,Black,Women's,Casual,Sandals,Black,Women's,Casual
4108,721468001,734460001,75,721468,734460,False,Sandals,Black,Women's,Casual,Sandals,Black,Women's,Casual


In [19]:
# 13. TAG PAIR TYPE & SUMMARY
# ============================================================
# Distinguishes "similar item" pairs (same type/color/gender/style,
# different product_code) from genuine cross-category complementary
# pairs — both are real signal, but they serve different use cases
# downstream (similar-item suggestions vs. true cross-sell).

cross_sell_final["is_similar_item"] = (
    (cross_sell_final["product_type_name_a"] == cross_sell_final["product_type_name_b"]) &
    (cross_sell_final["colour_group_name_a"] == cross_sell_final["colour_group_name_b"]) &
    (cross_sell_final["stylo_style_a"] == cross_sell_final["stylo_style_b"])
)

print("Similar-item pairs:", cross_sell_final["is_similar_item"].sum())
print("Genuine cross-category pairs:", (~cross_sell_final["is_similar_item"]).sum())

cross_sell_final.to_csv("cross_sell_pairs.csv", index=False)
print("\nSaved cross_sell_pairs.csv (final,", len(cross_sell_final), "rows)")

print("\nNOTEBOOK 03 SUMMARY")
print(f"Footwear articles analyzed: {len(shoes_mapped)}")
print(f"Color-popularity combinations with real signal: {len(color_popularity_final)}")
print(f"Documented gaps (using industry assumptions instead): Sports (all genders), Men's/Formal")
print(f"Shoe-purchase transactions: {len(shoe_transactions)}")
print(f"Final cross-sell pairs: {len(cross_sell_final)}")
print("Output files: color_popularity.csv, cross_sell_pairs.csv")
print("\n✓ Notebook 03 completed successfully.")

Similar-item pairs: 1715
Genuine cross-category pairs: 4707

Saved cross_sell_pairs.csv (final, 6422 rows)

NOTEBOOK 03 SUMMARY
Footwear articles analyzed: 5281
Color-popularity combinations with real signal: 178
Documented gaps (using industry assumptions instead): Sports (all genders), Men's/Formal
Shoe-purchase transactions: 745266
Final cross-sell pairs: 6422
Output files: color_popularity.csv, cross_sell_pairs.csv

✓ Notebook 03 completed successfully.


In [20]:
# 14. ZIP ARTIFACTS FOR LOCAL DOWNLOAD
# ============================================================
import shutil
from pathlib import Path

OUTPUT_FILES = [Path("color_popularity.csv"), Path("cross_sell_pairs.csv")]
ZIP_NAME = "notebook_03_hm_product_crosssell_artifact"
ARTIFACT_DIR = Path("notebook_03_outputs")

missing = [f for f in OUTPUT_FILES if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing output file(s): {missing}. Run the save cells first."
    )

ARTIFACT_DIR.mkdir(exist_ok=True)
for f in OUTPUT_FILES:
    shutil.copy(f, ARTIFACT_DIR / f.name)

zip_path = shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir=ARTIFACT_DIR.name)

print("ZIP created successfully:")
print(zip_path)
print(f"ZIP size: {Path(zip_path).stat().st_size / 1024:.2f} KB")
print(f"Contents: {[f.name for f in OUTPUT_FILES]}")

ZIP created successfully:
/kaggle/working/notebook_03_hm_product_crosssell_artifact.zip
ZIP size: 100.16 KB
Contents: ['color_popularity.csv', 'cross_sell_pairs.csv']
